In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf

X_train_seq = np.load('/kaggle/input/datasets/raedmuhammed/data600/X_train_seq.npy')
X_val_seq = np.load('/kaggle/input/datasets/raedmuhammed/data600/X_val_seq.npy')
y_train = np.load('/kaggle/input/datasets/raedmuhammed/data600/y_train.npy')
y_val = np.load('/kaggle/input/datasets/raedmuhammed/data600/y_val.npy')
X_test_seq=np.load('/kaggle/input/datasets/raedmuhammed/data600/X_test_seq.npy')
y_test=np.load('/kaggle/input/datasets/raedmuhammed/data600/y_test.npy')



train_dataset = tf.data.Dataset.from_tensor_slices((X_train_seq, y_train))
train_dataset = train_dataset.cache().shuffle(len(X_train_seq)).batch(16).prefetch(8)

val_dataset = tf.data.Dataset.from_tensor_slices((X_val_seq, y_val))
val_dataset = val_dataset.cache().batch(16).prefetch(8)

I0000 00:00:1790382505.606243      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1790382505.609487      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [2]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dropout, Dense, Embedding, GlobalMaxPooling1D
from tensorflow.keras.callbacks import EarlyStopping

classes = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

# handle imbalance in the data by use weighted_binary_crossentropy

pos_counts = y_train.sum(axis=0)
neg_counts = y_train.shape[0] - pos_counts

# define class weights 
class_weights = np.sqrt(neg_counts / pos_counts)

def weighted_binary_crossentropy(class_weights):
    class_weights = tf.constant(class_weights, dtype=tf.float32)
    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        y_true = tf.cast(y_true, tf.float32)
        bce = -(y_true * tf.math.log(y_pred) * class_weights +
                (1 - y_true) * tf.math.log(1 - y_pred))
        return tf.reduce_mean(bce)
    return loss


model = Sequential()

model.add(Embedding(100000, 120)) 
model.add((SimpleRNN(60, activation='tanh', return_sequences=True )))
model.add(GlobalMaxPooling1D())
model.add(Dropout(0.10))
model.add(Dense(50, activation='relu'))
model.add(Dropout(0.10))

# multi label classification
model.add(Dense(6, activation='sigmoid'))


# precision/recall per class to track performance individually
per_class_metrics = []
for i, class_name in enumerate(classes):
    per_class_metrics.append(
        tf.keras.metrics.Precision(class_id=i, name=f'precision_{class_name}')
    )
    per_class_metrics.append(
        tf.keras.metrics.Recall(class_id=i, name=f'recall_{class_name}')
    )

model.compile(
    loss=weighted_binary_crossentropy(class_weights),
    optimizer='Adam',
    metrics=per_class_metrics
)

model.summary()

# early stoping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

# train
history = model.fit(
    train_dataset,
    epochs=5,
    validation_data=val_dataset,
    callbacks=[early_stop]
)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
  16/7979 ━━━━━━━━━━━━━━━━━━━━ 1:26 11ms/step - loss: 0.7515 - precision_identity_hate: 0.0000e+00 - precision_insult: 0.0000e+00 - precision_obscene: 0.3333 - precision_severe_toxic: 0.0000e+00 - precision_threat: 0.0076 - precision_toxic: 0.0912 - recall_identity_hate: 0.0000e+00 - recall_insult: 0.0000e+00 - recall_obscene: 0.2085 - recall_severe_toxic: 0.0000e+00 - recall_threat: 0.8750 - recall_toxic: 0.6669      

I0000 00:00:1790382515.624315     128 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


7979/7979 ━━━━━━━━━━━━━━━━━━━━ 78s 9ms/step - loss: 0.1868 - precision_identity_hate: 0.1360 - precision_insult: 0.5353 - precision_obscene: 0.6166 - precision_severe_toxic: 0.2357 - precision_threat: 0.0403 - precision_toxic: 0.6802 - recall_identity_hate: 0.5294 - recall_insult: 0.7839 - recall_obscene: 0.8210 - recall_severe_toxic: 0.7898 - recall_threat: 0.1821 - recall_toxic: 0.7201 - val_loss: 0.1470 - val_precision_identity_hate: 0.1478 - val_precision_insult: 0.5354 - val_precision_obscene: 0.6161 - val_precision_severe_toxic: 0.3196 - val_precision_threat: 0.8571 - val_precision_toxic: 0.6814 - val_recall_identity_hate: 0.7295 - val_recall_insult: 0.8635 - val_recall_obscene: 0.9166 - val_recall_severe_toxic: 0.8219 - val_recall_threat: 0.0545 - val_recall_toxic: 0.8382
Epoch 2/5
7979/7979 ━━━━━━━━━━━━━━━━━━━━ 66s 8ms/step - loss: 0.1236 - precision_identity_hate: 0.2135 - precision_insult: 0.5628 - precision_obscene: 0.6795 - precision_severe_toxic: 0.2775 - precision_threat:

In [4]:
from sklearn.metrics import precision_recall_curve, f1_score, classification_report
import numpy as np

label_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

y_val_pred_prob = model.predict(val_dataset)

best_thresholds = []

# find best threshold per class using precision-recall curve (max F1)
for i, label in enumerate(label_cols):
    precisions, recalls, thresholds = precision_recall_curve(y_val[:, i], y_val_pred_prob[:, i])
    
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
    
    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
    best_f1 = f1_scores[best_idx]
    
    best_thresholds.append(best_threshold)
    print(f"{label}: best_threshold={best_threshold:.3f} | precision={precisions[best_idx]:.3f} | recall={recalls[best_idx]:.3f} | F1={best_f1:.3f}")

best_thresholds = np.array(best_thresholds)
print("\nFinal thresholds (derived from validation):", best_thresholds)

# apply tuned thresholds to get final predictions
y_val_pred_tuned = np.zeros_like(y_val_pred_prob, dtype=int)
for i in range(len(label_cols)):
    y_val_pred_tuned[:, i] = (y_val_pred_prob[:, i] > best_thresholds[i]).astype(int)

f1_per_class_tuned = f1_score(y_val, y_val_pred_tuned, average=None)
print("\n=== Validation results after threshold tuning ===")
for label, score in zip(label_cols, f1_per_class_tuned):
    print(f"F1 ({label}): {score:.4f}")

f1_macro_tuned = f1_score(y_val, y_val_pred_tuned, average='macro')
f1_micro_tuned = f1_score(y_val, y_val_pred_tuned, average='micro')
print(f"\nF1 Macro (val tuned): {f1_macro_tuned:.4f}")
print(f"F1 Micro (val tuned): {f1_micro_tuned:.4f}")

print("\n", classification_report(y_val, y_val_pred_tuned, target_names=label_cols))

1995/1995 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step
toxic: best_threshold=0.619 | precision=0.815 | recall=0.756 | F1=0.785
severe_toxic: best_threshold=0.716 | precision=0.395 | recall=0.697 | F1=0.505
obscene: best_threshold=0.806 | precision=0.813 | recall=0.788 | F1=0.800
threat: best_threshold=0.716 | precision=0.277 | recall=0.418 | F1=0.333
insult: best_threshold=0.671 | precision=0.625 | recall=0.792 | F1=0.699
identity_hate: best_threshold=0.818 | precision=0.450 | recall=0.495 | F1=0.471

Final thresholds (derived from validation): [0.61865854 0.7155547  0.80573386 0.7161493  0.67096406 0.8182171 ]

=== Validation results after threshold tuning ===
F1 (toxic): 0.7845
F1 (severe_toxic): 0.5028
F1 (obscene): 0.7998
F1 (threat): 0.3273
F1 (insult): 0.6984
F1 (identity_hate): 0.4686

F1 Macro (val tuned): 0.5969
F1 Micro (val tuned): 0.7283

                precision    recall  f1-score   support

        toxic       0.82      0.76      0.78      3059
 severe_toxic       0.39      0.69   

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [5]:

#Evaluating on the test data set

y_pred_prob = model.predict(X_test_seq, batch_size=32)

thresholds = {
    'toxic': best_thresholds[0],
    'severe_toxic': best_thresholds[1],
    'obscene': best_thresholds[2],
    'threat': best_thresholds[3],
    'insult': best_thresholds[4],
    'identity_hate':best_thresholds[5]
}

y_pred = np.zeros_like(y_pred_prob, dtype=int)
for i, label in enumerate(label_cols):
    y_pred[:, i] = (y_pred_prob[:, i] > thresholds[label]).astype(int)

f1_per_class = f1_score(y_test, y_pred, average=None)
for label, score in zip(label_cols, f1_per_class):
    print(f"F1 ({label}): {score:.4f}  (threshold={thresholds[label]})")

f1_macro = f1_score(y_test, y_pred, average='macro')
f1_micro = f1_score(y_test, y_pred, average='micro')
print(f"\nF1 Macro: {f1_macro:.4f}")
print(f"F1 Micro: {f1_micro:.4f}")

print("\n", classification_report(y_test, y_pred, target_names=label_cols))

2000/2000 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step
F1 (toxic): 0.6441  (threshold=0.6186585426330566)
F1 (severe_toxic): 0.3224  (threshold=0.7155547142028809)
F1 (obscene): 0.6822  (threshold=0.805733859539032)
F1 (threat): 0.2707  (threshold=0.7161492705345154)
F1 (insult): 0.5727  (threshold=0.6709640622138977)
F1 (identity_hate): 0.5318  (threshold=0.8182170987129211)

F1 Macro: 0.5040
F1 Micro: 0.6077

                precision    recall  f1-score   support

        toxic       0.52      0.85      0.64      6090
 severe_toxic       0.21      0.72      0.32       367
      obscene       0.62      0.76      0.68      3691
       threat       0.18      0.51      0.27       211
       insult       0.45      0.80      0.57      3427
identity_hate       0.48      0.60      0.53       712

    micro avg       0.49      0.79      0.61     14498
    macro avg       0.41      0.71      0.50     14498
 weighted avg       0.51      0.79      0.62     14498
  samples avg       0.07      0.07      0.07

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
